# Cost-Quality-Speed
# 0. 介绍

**研究背景**：Agent Harness 会决定使用哪个模型、提供多少上下文、调用哪些工具，以及是否执行校验、重试和恢复。这些机制可以提高任务质量，却也会增加 Token、API 费用和等待时间；生产系统还必须同时满足预算、响应时限和风险要求，因此成本、质量与速度需要放在一起衡量。

**现存问题**：生产中常见的错误基线是让所有任务共用一条固定的低成本快速路径：始终选择便宜模型，跳过结果校验和失败恢复，并把 API 正常返回误当成任务成功。它在平均延迟和单次费用上看起来更好，却会让困难或高风险任务带着错误结果进入真实环境。反过来，让所有任务无条件使用最强模型和最严检查，也会造成费用失控、响应超时和资源浪费。只看单一平均分或把多个指标压成一个总分，都无法说明哪种配置真正适合当前约束。

**解决方案**：本 Notebook 将实现一个极简的 Cost-Quality-Speed Harness，采用`同口径评测 + 风险与预算感知路由 + 按需升级 + 约束下 Pareto 选择`机制：先让 `fast`、`cheap`、`safe` 三种配置完成同一批任务，分别记录成功率、风险事件、Token、成本和延迟；运行时让简单低风险任务走轻量路径，只在任务困难、风险较高或校验失败时升级模型、检查与重试；最后先淘汰不满足质量、风险、预算或时限硬约束的配置，再从 Pareto 前沿选择合适方案。后文会用同一真实 API 对比固定快速基线因跳过校验而失败，与改进版本按约束选择安全路径后成功，从而直观看到可靠优化不是一味追求更便宜、更快或更强，而是让每项额外开销只用在确实需要它的任务上。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 固定共同任务
成本、质量和速度只有在处理同一任务时才能公平比较。下面固定一笔高额退款请求：金额高于常见的自动处理范围，后续三种 Harness 配置都会读取完全相同的输入。

In [2]:
# request_id 让后续每条结果都能对应到同一笔请求
# amount 和 risk_level 是选择处理路径时最重要的两个事实
refund_request = {
    "request_id": "R-104",
    "amount": 1200,
    "reason": "重复扣款",
    "risk_level": "high",
}

print("退款请求：", refund_request)

退款请求： {'request_id': 'R-104', 'amount': 1200, 'reason': '重复扣款', 'risk_level': 'high'}


输出显示了后续实验唯一使用的退款请求。请求金额为 `1200`，风险等级为 `high`；下一步准备模型训练数据中没有的当前退款规则。

## 2.2 准备实时规则工具
大模型不能可靠记住企业当前使用的退款额度。下面定义一个极简的规则查询工具：程序调用它时，才能取得本次实验采用的自动退款上限。

In [3]:
# auto_approve_limit 表示程序可以自动退款的最高金额
# 超过额度的请求必须转人工复核，不能直接自动退款
refund_policy = {
    "auto_approve_limit": 500,
    "above_limit_action": "manual_review",
}

def get_refund_policy():
    # 返回副本，避免调用者改动原始规则
    return refund_policy.copy()

print("当前退款规则：", get_refund_policy())

当前退款规则： {'auto_approve_limit': 500, 'above_limit_action': 'manual_review'}


输出表明自动退款上限是 `500`，超过上限必须转人工复核。`1200` 高于这个额度，因此后续正确路径必须使用这条实时规则；下一步规定真实模型第一次只提交哪些草稿字段。

## 2.3 规定模型初稿格式
第一次模型调用只负责读懂请求，不负责查询规则或作出最终决定。下面用结构化工具要求模型提交请求编号、金额和问题摘要，使三种配置从同一份真实模型初稿开始。

In [4]:
# 工具只收集原始请求中的事实，不包含最终退款决定
# required 固定三种 Harness 配置共同使用的初稿字段
draft_tools = [{
    "type": "function",
    "function": {
        "name": "submit_refund_draft",
        "description": "提交退款请求的初步摘要",
        "parameters": {
            "type": "object",
            "properties": {
                "request_id": {"type": "string"},
                "amount": {"type": "integer"},
                "summary": {"type": "string"},
            },
            "required": ["request_id", "amount", "summary"],
        },
    },
}]

print("初稿工具：", draft_tools[0]["function"]["name"])
print("初稿字段：", draft_tools[0]["function"]["parameters"]["required"])

初稿工具： submit_refund_draft
初稿字段： ['request_id', 'amount', 'summary']


输出显示初稿只有三个事实字段，没有规则额度和最终动作。初稿本身不是错误，把它未经处理就当成最终结果才是错误；下一步固定三种 Harness 配置的差异。

## 2.4 固定三种 Harness 配置
三种配置只改变外层程序投入的工作量。`fast` 直接使用初稿，`cheap` 额外查询本地规则，`safe` 在查询规则后再让真实模型修正一次最终结果。

In [5]:
# policy_lookup 表示是否为当前请求读取实时业务规则
# repair_calls 表示初稿之后还要增加几次真实模型调用
harness_variants = [
    {"name": "fast", "policy_lookup": False, "repair_calls": 0},
    {"name": "cheap", "policy_lookup": True, "repair_calls": 0},
    {"name": "safe", "policy_lookup": True, "repair_calls": 1},
]

for variant in harness_variants:
    print(variant)

{'name': 'fast', 'policy_lookup': False, 'repair_calls': 0}
{'name': 'cheap', 'policy_lookup': True, 'repair_calls': 0}
{'name': 'safe', 'policy_lookup': True, 'repair_calls': 1}


输出列出了后续对照使用的三种配置。它们共享模型、任务和初稿，区别只在是否查询规则、是否增加修正调用；下一步固定所有配置共同面对的正确终态。

## 2.5 固定成功标准
`1200` 高于自动退款上限 `500`，所以最终结果必须保留请求事实、写明当前额度，并把处理动作设为人工复核。后续无论配置多快或多省，都使用这一份标准判断质量。

In [6]:
# expected_result 固定三种配置共同使用的唯一正确终态
# final_action 直接来自实时规则，不由模型自行猜测
expected_result = {
    "request_id": refund_request["request_id"],
    "amount": refund_request["amount"],
    "policy_limit": refund_policy["auto_approve_limit"],
    "final_action": refund_policy["above_limit_action"],
}

print("成功标准：", expected_result)

成功标准： {'request_id': 'R-104', 'amount': 1200, 'policy_limit': 500, 'final_action': 'manual_review'}


输出给出了唯一正确终态。至此，共同任务、实时规则、模型初稿格式、三种配置和成功标准都已固定；下一章将调用真实 API 取得一份共同初稿，并保存 Token、延迟和停止原因。

# 3. 获取并验证 API 响应
## 3.1 准备模型可见消息
三种 Harness 配置必须从同一份模型初稿开始。下面只把第 2 章固定的退款请求交给模型，并明确要求它提取事实、不查询规则、不作最终退款决定。

In [7]:
# system 消息限制本次调用只生成初稿，不越过 Harness 作最终决定
# user 消息直接复用第 2 章的请求，三种配置不会得到不同输入
draft_messages = [
    {
        "role": "system",
        "content": "只提取退款请求事实，并调用 submit_refund_draft。不要查询规则，不要作最终退款决定。",
    },
    {
        "role": "user",
        "content": (
            f"请求编号：{refund_request['request_id']}；"
            f"退款金额：{refund_request['amount']}；"
            f"退款原因：{refund_request['reason']}。"
        ),
    },
]

print("模型任务：", draft_messages[-1]["content"])

模型任务： 请求编号：R-104；退款金额：1200；退款原因：重复扣款。


输出显示模型只会看到请求编号、金额和原因，不会看到退款额度。这样取得的是事实初稿，不是最终决定；下一步把消息和初稿工具发送给真实 API。

## 3.2 发送真实 API 请求
下面使用第 1 章创建的客户端发送一次请求，并记录实际等待时间。请求强制使用第 2 章的初稿工具，因此模型返回的是程序可以继续处理的结构化调用。

In [8]:
from time import perf_counter

# 计时从请求发出前开始，包含等待 provider 返回的时间
# temperature 设为 0，让共同初稿尽量稳定且容易复现
draft_started = perf_counter()
draft_response = client.chat.completions.create(
    model=model_name,
    messages=draft_messages,
    tools=draft_tools,
    tool_choice="required",
    temperature=0,
)
draft_latency_ms = round((perf_counter() - draft_started) * 1000)

print("真实 API 初稿已收到")

真实 API 初稿已收到


输出说明真实 API 已经返回，完整响应保存在 `draft_response` 中。此时规则工具仍未执行，也没有产生最终退款动作；下一步读取模型提交的工具名称和初稿参数。

## 3.3 查看并保存模型初稿
工具参数是 JSON 文本。下面把它还原成普通字典，并保留调用编号、工具名称和三个初稿字段，供后面的三种 Harness 配置共同使用。

In [9]:
import json

# 第一条 choice 保存本次真实请求产生的模型决定
# arguments 是 JSON 字符串，需要还原成后续代码可读取的字典
draft_choice = draft_response.choices[0]
draft_tool_call = draft_choice.message.tool_calls[0]
draft_call_id = draft_tool_call.id
refund_draft = json.loads(draft_tool_call.function.arguments)

print("调用编号：", draft_call_id)
print("工具名称：", draft_tool_call.function.name)
print("模型初稿：", refund_draft)

调用编号： call_7a9d836ff5dd43f8bb4275db
工具名称： submit_refund_draft
模型初稿： {'request_id': 'R-104', 'amount': 1200, 'summary': '客户反馈遭遇重复扣款问题，申请退款1200元。需要核实交易记录确认是否存在重复扣款情况。'}


输出显示真实模型正确提取了请求编号、金额和问题摘要，说明响应可以被程序读取。初稿仍然没有规则额度和最终动作，因此不能直接视为任务成功；下一步保存本次调用的实际运行指标。

## 3.4 查看本次请求信息
成本、质量和速度必须使用真实运行数据比较。下面读取 provider 返回的 Token 和停止原因，再与模型名称、调用次数和实测等待时间放在同一个字典中。

In [10]:
# usage 来自真实 API 响应，不用字符数估算 Token
# provider 没有返回本次账单金额，因此美元成本明确保留为空值
draft_usage = draft_response.usage
draft_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "api_calls": 1,
    "input_tokens": draft_usage.prompt_tokens,
    "output_tokens": draft_usage.completion_tokens,
    "total_tokens": draft_usage.total_tokens,
    "cost_usd": None,
    "latency_ms": draft_latency_ms,
    "stop_reason": draft_choice.finish_reason,
}

print(draft_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'api_calls': 1, 'input_tokens': 215, 'output_tokens': 186, 'total_tokens': 401, 'cost_usd': None, 'latency_ms': 5631, 'stop_reason': 'tool_calls'}


输出记录了本次真实调用的 provider、模型、调用次数、Token、成本状态、延迟和停止原因。`tool_calls` 只表示模型提交了结构化初稿，不表示退款任务已经完成；下一章将定义把这份初稿直接当成最终结果的快速基线组件。

# 4. 定义基线组件 *
## 4.1 定义固定快速路径
生产中常见的错误基线是把第一次模型正常返回当成任务完成，再附上一个固定动作直接进入下游。下面定义最小的 `fast` 组件：它复制第 3 章的真实初稿并默认自动退款，不查询当前额度，也不增加修正调用。

In [11]:
# 复制初稿，保留真实模型提取的请求事实
# 固定动作追求最短路径，不读取退款规则，也不请求模型修正
def finalize_fast(draft):
    final_result = draft.copy()
    final_result["final_action"] = "auto_approve"
    return final_result

print("基线组件：固定快速路径")

基线组件：固定快速路径


输出说明快速基线已经定义，但还没有处理第 3 章的真实初稿。这个组件只做一次复制和一次默认赋值，速度快、额外模型成本为零；下一章将运行它，并与第 2 章的成功标准直接比较。

# 5. 展示基线故障 *
## 5.1 运行固定快速路径
现在把第 3 章保存的真实模型初稿交给 `fast`。下面同时记录这段本地 Harness 逻辑的耗时，观察它用多短的路径生成了什么最终结果。

In [12]:
# 计时只覆盖快速 Harness，不重复计算第 3 章的 API 等待时间
# 输入直接使用同一次真实模型初稿，不重新请求模型
fast_started = perf_counter()
fast_result = finalize_fast(refund_draft)
fast_harness_latency_ms = round((perf_counter() - fast_started) * 1000, 4)

print("基线结果：", fast_result)
print("Harness 耗时：", fast_harness_latency_ms, "ms")

基线结果： {'request_id': 'R-104', 'amount': 1200, 'summary': '客户反馈遭遇重复扣款问题，申请退款1200元。需要核实交易记录确认是否存在重复扣款情况。', 'final_action': 'auto_approve'}
Harness 耗时： 0.0006 ms


输出显示快速路径几乎立刻给出了结果，但其中没有当前退款额度，并把高额请求设为 `auto_approve`。速度快只说明步骤少，不能说明结果正确；下一步定义前后两条路径共用的评分方式。

## 5.2 定义统一评分方式
为了避免给基线和改进版本使用不同标准，下面逐项比较结果与第 2 章的正确终态。质量分是正确字段所占比例，四项全部正确才算任务成功。

In [13]:
# target 中的每个字段都使用完全相同的等值比较
# quality_score 只表示正确字段比例，不能替代完整成功判定
def grade_result(result, target):
    checks = {}
    for field in target:
        checks[field] = result.get(field) == target[field]

    quality_score = round(sum(checks.values()) / len(checks), 2)
    return {
        "checks": checks,
        "quality_score": quality_score,
        "success": all(checks.values()),
    }

print("统一评分函数已定义")

统一评分函数已定义


输出说明统一评分函数已经准备好，但还没有评价任何配置。它只读取实际结果和固定终态，不关心结果来自快速路径还是改进路径；下一步用它判断基线。

## 5.3 判断基线结果
下面把刚才得到的基线结果交给统一评分函数，并逐项打印四个字段。这样可以直接看到快速路径保留了哪些事实，又漏掉或做错了什么。

In [14]:
# 基线与后续改进版本都使用同一个 expected_result
# 逐项打印比只给一个总分更容易定位失败原因
fast_grade = grade_result(fast_result, expected_result)

for field, passed in fast_grade["checks"].items():
    print(field, "：", passed)

print("质量分：", fast_grade["quality_score"])
print("任务成功：", fast_grade["success"])

request_id ： True
amount ： True
policy_limit ： False
final_action ： False
质量分： 0.5
任务成功： False


输出表明请求编号和金额正确，但规则额度缺失，最终动作也错误，因此质量分只有 `0.5`，完整任务失败。真实模型初稿中的事实没有错，故障来自 Harness 把初稿直接变成了最终决定；下一步量化这条路径省下了什么、付出了什么。

## 5.4 汇总基线指标
成本-质量-速度不能只看其中一项。下面把第 3 章的真实 API 用量与本地 Harness 耗时合并，并记录错误自动放行产生的一次风险事件。

In [15]:
# fast 只使用共同初稿调用，因此 API 次数和 Token 不再增加
# 高额请求被自动放行，记为一次直接影响业务的风险事件
fast_risk_events = 1
fast_metrics = {
    "variant": "fast",
    "quality_score": fast_grade["quality_score"],
    "success": fast_grade["success"],
    "api_calls": draft_metrics["api_calls"],
    "total_tokens": draft_metrics["total_tokens"],
    "cost_usd": draft_metrics["cost_usd"],
    "latency_ms": round(draft_metrics["latency_ms"] + fast_harness_latency_ms, 2),
    "risk_events": fast_risk_events,
}

print(fast_metrics)

{'variant': 'fast', 'quality_score': 0.5, 'success': False, 'api_calls': 1, 'total_tokens': 401, 'cost_usd': None, 'latency_ms': 5631.0, 'risk_events': 1}


输出显示 `fast` 只用一次 API 调用，Token 和延迟都最低，但任务失败并产生一次风险事件。这就是固定快速路径的核心问题：它把省下来的检查与修正成本转化成了错误结果；下一章将定义按风险查询规则并按需升级的改进组件。

# 6. 定义改进组件 *
## 6.1 定义风险感知路由
当前学术与工业系统的共同做法不是让所有请求固定走同一条路径，而是按任务风险决定投入多少资源。下面使用最小三档规则：低风险走 `fast`，中风险走 `cheap`，高风险走带一次修正调用的 `safe`。

In [16]:
# high 请求优先满足质量约束，因此选择完整 safe 路径
# medium 只增加本地规则查询，low 保留最短 fast 路径
def choose_variant(request):
    if request["risk_level"] == "high":
        return "safe"
    if request["risk_level"] == "medium":
        return "cheap"
    return "fast"

print("改进组件：风险感知路由")

改进组件：风险感知路由


输出说明路由器已经定义，但还没有读取当前请求。它不会让每个任务都支付最高成本，只把高风险任务升级到 `safe`；下一步定义只增加本地规则查询的 `cheap` 路径。

## 6.2 定义规则补全路径
`cheap` 用一次本地工具查询换取更完整的上下文，但不追加模型调用。下面让它在快速结果中补上当前退款额度，同时保留原来的默认动作，便于后续观察“局部质量提高”是否等于完整成功。

In [17]:
# 先复用 fast 的同一份真实初稿和默认动作
# 再调用本地规则工具，只补充当前自动退款额度
def finalize_cheap(draft):
    current_policy = get_refund_policy()
    final_result = finalize_fast(draft)
    final_result["policy_limit"] = current_policy["auto_approve_limit"]
    return final_result

print("改进组件：规则补全路径")

改进组件：规则补全路径


输出说明 `cheap` 已经定义。它能用极低的本地开销补上规则额度，却不会让模型依据规则重新决定最终动作；下一步规定 `safe` 修正调用必须提交的完整终稿格式。

## 6.3 规定完整终稿格式
修正调用不能再返回缺少关键字段的初稿。下面用结构化工具要求模型一次提交请求编号、金额、当前规则额度和最终动作，使输出可以直接交给第 5 章的统一评分函数。

In [18]:
# 四个 required 字段与第 2 章的 expected_result 完全对应
# final_action 只允许自动退款或人工复核两个明确动作
final_tools = [{
    "type": "function",
    "function": {
        "name": "submit_refund_result",
        "description": "提交依据当前规则生成的最终退款结果",
        "parameters": {
            "type": "object",
            "properties": {
                "request_id": {"type": "string"},
                "amount": {"type": "integer"},
                "policy_limit": {"type": "integer"},
                "final_action": {
                    "type": "string",
                    "enum": ["auto_approve", "manual_review"],
                },
            },
            "required": ["request_id", "amount", "policy_limit", "final_action"],
        },
    },
}]

print("终稿工具：", final_tools[0]["function"]["name"])
print("终稿字段：", final_tools[0]["function"]["parameters"]["required"])

终稿工具： submit_refund_result
终稿字段： ['request_id', 'amount', 'policy_limit', 'final_action']


输出显示终稿必须包含四个可直接评分的字段。结构化格式只负责把决定说清楚，真正的正确动作仍需依据实时规则推导；下一步定义修正调用看到的消息。

## 6.4 定义按需修正消息
`safe` 只在路由器判定需要升级时才多调用一次模型。下面把同一份真实初稿和本地工具取得的当前规则放进消息，让模型根据金额与额度的关系生成完整终稿。

In [19]:
# draft 保留第一次真实调用提取的请求事实
# policy 来自本地工具，修正调用不依赖模型记忆企业规则
def build_safe_messages(draft, policy):
    return [
        {
            "role": "system",
            "content": "依据提供的当前规则完成退款决定，并调用 submit_refund_result。",
        },
        {
            "role": "user",
            "content": (
                f"初稿：{json.dumps(draft, ensure_ascii=False)}\n"
                f"自动退款上限：{policy['auto_approve_limit']}\n"
                f"超过上限的动作：{policy['above_limit_action']}"
            ),
        },
    ]

print("改进组件：按需修正消息")

改进组件：按需修正消息


输出说明完整改进链已经定义：先按风险选择配置，`cheap` 只查询本地规则，`safe` 再把规则反馈给真实模型并要求完整终稿。本章没有执行任何配置；下一章将运行三条路径并展示修复结果。

# 7. 展示修复结果 *
## 7.1 为当前任务选择路径
先让第 6 章的路由器读取第 2 章固定的风险等级。路由只决定投入哪种 Harness 配置，不修改请求，也不调用模型。

In [20]:
# 输入只使用任务原本携带的 risk_level
# high 应选择 safe，把额外成本留给高风险请求
selected_variant = choose_variant(refund_request)

print("风险等级：", refund_request["risk_level"])
print("选择路径：", selected_variant)

风险等级： high
选择路径： safe


输出显示高风险请求被分配到 `safe`，说明它不会沿用错误的固定快速路径。为了看清每增加一步带来的收益，下一步先运行只补本地规则的 `cheap`。

## 7.2 运行规则补全路径
`cheap` 不发送新的模型请求，只在第 3 章的同一份真实初稿上查询当前额度。下面记录这段本地逻辑的结果和耗时。

In [21]:
# 计时只覆盖本地规则查询与字段补全
# 输入继续复用同一份真实初稿，不增加 API 调用
cheap_started = perf_counter()
cheap_result = finalize_cheap(refund_draft)
cheap_harness_latency_ms = round((perf_counter() - cheap_started) * 1000, 4)

print("cheap 结果：", cheap_result)
print("Harness 耗时：", cheap_harness_latency_ms, "ms")

cheap 结果： {'request_id': 'R-104', 'amount': 1200, 'summary': '客户反馈遭遇重复扣款问题，申请退款1200元。需要核实交易记录确认是否存在重复扣款情况。', 'final_action': 'auto_approve', 'policy_limit': 500}
Harness 耗时： 0.0011 ms


输出显示 `cheap` 已补上 `policy_limit=500`，却仍保留 `auto_approve`。这说明增加上下文只会让信息更完整，不会自动让旧决定变正确；下一步用统一标准量化这次局部提升。

## 7.3 判断规则补全结果
下面继续使用第 5 章的同一个评分函数。`cheap` 如果只修好一个字段，质量分应该提高，但完整任务仍不能通过。

In [22]:
# 评分目标仍是第 2 章固定的 expected_result
# 逐项输出用于区分局部改善和完整任务成功
cheap_grade = grade_result(cheap_result, expected_result)

for field, passed in cheap_grade["checks"].items():
    print(field, "：", passed)

print("质量分：", cheap_grade["quality_score"])
print("任务成功：", cheap_grade["success"])

request_id ： True
amount ： True
policy_limit ： True
final_action ： False
质量分： 0.75
任务成功： False


输出表明 `cheap` 的质量从 `0.5` 提高到 `0.75`，但错误动作使任务仍然失败。这就是“局部变好不等于整体完成”；下一步为路由器选中的 `safe` 准备实时规则上下文。

## 7.4 准备安全路径上下文
`safe` 需要让模型真正依据规则改写决定。下面取得当前规则，并用第 6 章的消息构造器把它与同一份真实初稿放进修正请求。

In [23]:
# current_policy 来自第 2 章定义的本地工具
# safe_messages 同时保留原始初稿和当前规则事实
current_policy = get_refund_policy()
safe_messages = build_safe_messages(refund_draft, current_policy)

print("实时规则：", current_policy)
print("修正输入：")
print(safe_messages[-1]["content"])

实时规则： {'auto_approve_limit': 500, 'above_limit_action': 'manual_review'}
修正输入：
初稿：{"request_id": "R-104", "amount": 1200, "summary": "客户反馈遭遇重复扣款问题，申请退款1200元。需要核实交易记录确认是否存在重复扣款情况。"}
自动退款上限：500
超过上限的动作：manual_review


输出显示修正请求同时包含初稿、自动退款上限和超额动作。模型不需要猜测企业规则；下一步发送本 Notebook 的第二次真实 API 请求。

## 7.5 获取真实修正终稿
下面把修正上下文与完整终稿工具发送给同一个真实模型。相比 `fast` 和 `cheap`，这一步明确增加一次 API 调用，并记录新增的等待时间。

In [24]:
# 计时只覆盖新增的修正调用，便于与共同初稿成本相加
# tool_choice 要求模型通过完整终稿协议提交最终决定
safe_started = perf_counter()
safe_response = client.chat.completions.create(
    model=model_name,
    messages=safe_messages,
    tools=final_tools,
    tool_choice="required",
    temperature=0,
)
safe_api_latency_ms = round((perf_counter() - safe_started) * 1000)

print("真实 API 修正终稿已收到")

真实 API 修正终稿已收到


输出说明第二次真实 API 请求已经返回。额外等待时间是 `safe` 为高风险任务支付的速度成本；下一步读取结构化终稿和调用编号。

## 7.6 查看并保存修正终稿
终稿参数仍是 JSON 文本。下面把它还原为字典，并保存第二次工具调用的编号、名称和完整结果。

In [25]:
# 第一条 choice 是模型依据实时规则作出的修正决定
# arguments 还原后可以直接交给统一评分函数
safe_choice = safe_response.choices[0]
safe_tool_call = safe_choice.message.tool_calls[0]
safe_call_id = safe_tool_call.id
safe_result = json.loads(safe_tool_call.function.arguments)

print("调用编号：", safe_call_id)
print("工具名称：", safe_tool_call.function.name)
print("safe 结果：", safe_result)

调用编号： call_f2c18de6b7f64f5c83a999f2
工具名称： submit_refund_result
safe 结果： {'request_id': 'R-104', 'amount': 1200, 'policy_limit': 500, 'final_action': 'manual_review'}


输出显示 `safe` 已保留请求事实、写入额度 `500`，并把超额请求改为 `manual_review`。结果看起来正确，但仍要使用与基线完全相同的标准判断；下一步执行统一评分。

## 7.7 判断安全路径结果
下面把真实模型修正后的结构化终稿交给第 5 章的统一评分函数。只有四项全部通过，`safe` 才能证明修复有效。

In [26]:
# safe 继续使用同一个 expected_result，没有单独放宽标准
# 逐项输出直接展示修正调用解决了哪些字段
safe_grade = grade_result(safe_result, expected_result)

for field, passed in safe_grade["checks"].items():
    print(field, "：", passed)

print("质量分：", safe_grade["quality_score"])
print("任务成功：", safe_grade["success"])

request_id ： True
amount ： True
policy_limit ： True
final_action ： True
质量分： 1.0
任务成功： True


输出中四项全部为 `True`，质量分达到 `1.0`，任务成功。模型没有变化，差异来自 Harness 为高风险请求补充实时规则并按需增加一次修正调用；最后汇总 `cheap` 与 `safe` 的代价。

## 7.8 汇总改进路径指标
下面沿用第 5 章的指标口径。`cheap` 只承担共同初稿成本，`safe` 还要加上修正调用的真实 Token 和延迟；错误自动放行计为风险事件。

In [27]:
# cheap 不增加 API 调用，但错误动作仍产生一次风险事件
# safe 把两次真实调用的 Token 和等待时间按同一口径相加
safe_usage = safe_response.usage
cheap_metrics = {
    "variant": "cheap",
    "quality_score": cheap_grade["quality_score"],
    "success": cheap_grade["success"],
    "api_calls": draft_metrics["api_calls"],
    "total_tokens": draft_metrics["total_tokens"],
    "cost_usd": draft_metrics["cost_usd"],
    "latency_ms": round(draft_metrics["latency_ms"] + cheap_harness_latency_ms, 2),
    "risk_events": 1,
}
safe_metrics = {
    "variant": "safe",
    "quality_score": safe_grade["quality_score"],
    "success": safe_grade["success"],
    "api_calls": draft_metrics["api_calls"] + 1,
    "total_tokens": draft_metrics["total_tokens"] + safe_usage.total_tokens,
    "cost_usd": None,
    "latency_ms": draft_metrics["latency_ms"] + safe_api_latency_ms,
    "risk_events": 0,
}

print(cheap_metrics)
print(safe_metrics)

{'variant': 'cheap', 'quality_score': 0.75, 'success': False, 'api_calls': 1, 'total_tokens': 401, 'cost_usd': None, 'latency_ms': 5631.0, 'risk_events': 1}
{'variant': 'safe', 'quality_score': 1.0, 'success': True, 'api_calls': 2, 'total_tokens': 823, 'cost_usd': None, 'latency_ms': 10252, 'risk_events': 0}


输出显示 `cheap` 不增加模型成本却只得到局部改善，`safe` 以更多 Token 和更长延迟换来满分与零风险。三条路径的真实结果已经齐全；下一章将把它们放入同一张消融表和 Pareto 前沿中比较。

# 8. 汇总消融对照
三条路径使用同一任务、同一模型初稿和同一评分标准，因此可以直接比较。下面先保留每项原始指标，再判断谁被全面支配；不会把质量、Token、延迟和风险压成一个主观总分。

## 8.1 打印同口径消融表
下面把三组指标按固定列排成一张表。当前 `.env` 没有提供价格表，所以美元成本继续显示为“未配置”，并用真实 `total_tokens` 作为成本代理。

In [28]:
# 三行只引用前文真实运行得到的指标，不重新估算任何数值
# 固定列顺序让质量、资源和风险可以逐项横向比较
ablation_rows = [fast_metrics, cheap_metrics, safe_metrics]
ablation_columns = [
    "variant",
    "quality_score",
    "success",
    "api_calls",
    "total_tokens",
    "cost_usd",
    "latency_ms",
    "risk_events",
]

print("消融对照：")
print(" | ".join(ablation_columns))
for row in ablation_rows:
    displayed_values = []
    for column in ablation_columns:
        value = row[column]
        if column == "cost_usd" and value is None:
            value = "未配置"
        displayed_values.append(str(value))
    print(" | ".join(displayed_values))

消融对照：
variant | quality_score | success | api_calls | total_tokens | cost_usd | latency_ms | risk_events
fast | 0.5 | False | 1 | 401 | 未配置 | 5631.0 | 1
cheap | 0.75 | False | 1 | 401 | 未配置 | 5631.0 | 1
safe | 1.0 | True | 2 | 823 | 未配置 | 10252 | 0


表中 `cheap` 与 `fast` 的 Token、API 次数和延迟相同，却把质量从 `0.5` 提高到 `0.75`；这说明读取本地规则的轻量构件带来了免费改进，但二者仍然失败并各有一次风险事件。`safe` 达到满分和零风险，代价是第二次真实 API 调用、更多 Token 和更长等待。

## 8.2 定义 Pareto 支配关系
Pareto 判断问的是：一个方案能否在所有目标上都不差，并且至少一项更好。这里最大化质量，最小化 Token、延迟和风险；`success` 留给最后的发布硬约束，不重复算进支配目标。

In [29]:
# no_worse 要求左侧配置在四个目标上都不劣于右侧配置
# strictly_better 防止完全相同的两个点互相支配
def dominates(left, right):
    no_worse = (
        left["quality_score"] >= right["quality_score"]
        and left["total_tokens"] <= right["total_tokens"]
        and left["latency_ms"] <= right["latency_ms"]
        and left["risk_events"] <= right["risk_events"]
    )
    strictly_better = (
        left["quality_score"] > right["quality_score"]
        or left["total_tokens"] < right["total_tokens"]
        or left["latency_ms"] < right["latency_ms"]
        or left["risk_events"] < right["risk_events"]
    )
    return no_worse and strictly_better

print("cheap 支配 fast：", dominates(cheap_metrics, fast_metrics))
print("safe 支配 cheap：", dominates(safe_metrics, cheap_metrics))

cheap 支配 fast： True
safe 支配 cheap： False


`cheap` 支配 `fast`，因为资源和风险不变而质量更高；继续保留 `fast` 没有收益。`safe` 不支配 `cheap`，因为它虽然质量更高、风险更低，却更慢且使用更多 Token。两者之间存在真实取舍。

## 8.3 计算 Pareto 前沿
下面让每个候选与其他候选逐一比较。只要存在另一个配置支配它，就把它排除；没有被任何配置支配的点共同构成 Pareto 前沿。

In [30]:
# 外层循环依次检查每个候选是否值得保留
# 内层循环显式寻找能够全面支配当前候选的配置
pareto_front = []
for candidate in ablation_rows:
    is_dominated = False
    for challenger in ablation_rows:
        if challenger["variant"] == candidate["variant"]:
            continue
        if dominates(challenger, candidate):
            is_dominated = True
            break
    if not is_dominated:
        pareto_front.append(candidate)

pareto_variants = []
for row in pareto_front:
    pareto_variants.append(row["variant"])

print("Pareto 前沿：", pareto_variants)

Pareto 前沿： ['cheap', 'safe']


前沿保留 `cheap` 和 `safe`：前者节省资源，后者保证结果与风险，没有一个能在四项指标上全面胜过另一个。Pareto 前沿只说明“值得考虑”，还没有结合当前退款任务的发布条件。

## 8.4 应用发布硬约束
本任务涉及高风险退款，生产发布要求必须成功且风险事件为零。下面先用这两个条件过滤 Pareto 前沿，再在可发布候选中选择 Token 最少的配置。

In [31]:
# 发布门禁先排除任务失败或仍产生风险事件的配置
# 若有多个候选通过门禁，再逐个选择 Token 最少者
release_candidates = []
for row in pareto_front:
    passes_success = row["success"] is True
    passes_risk = row["risk_events"] == 0
    if passes_success and passes_risk:
        release_candidates.append(row)

selected_variant = None
selected_tokens = None
for row in release_candidates:
    if selected_tokens is None or row["total_tokens"] < selected_tokens:
        selected_variant = row["variant"]
        selected_tokens = row["total_tokens"]

release_variants = []
for row in release_candidates:
    release_variants.append(row["variant"])

print("通过发布门禁：", release_variants)
print("最终选择：", selected_variant)

通过发布门禁： ['safe']
最终选择： safe


只有 `safe` 同时满足“任务成功”和“零风险”，所以最终选择它。结论不是 `safe` 在所有场景都最好，而是高风险任务的硬约束让额外 Token 与延迟成为必要成本；低风险任务仍可按约束选择更轻的路径。


## 8.6 拓展
### nano 版省略了什么
nano 版省略了学习型路由器、多模型真实切换、provider 价格表、p50/p95/p99 延迟、重复 rollout 与置信区间、会话级预算、在线 A/B 实验和漂移监控。这些能力决定了生产结论能否跨任务、跨时间成立，但不改变本 Notebook 展示的最小机制：**先测量取舍，再用风险和预算约束选择 Harness，而不是把最快或最强配置当成固定答案。**

### 延伸阅读

1. 2024, [RouteLLM](https://arxiv.org/abs/2406.18665)：用偏好数据在强弱模型之间学习成本-质量路由。
2. 2024, [Towards Optimizing the Costs of LLM Usage](https://arxiv.org/abs/2402.01742)：缓存、级联与路由等成本优化方法。
3. 2026, [Dual-Pool Token-Budget Routing](https://arxiv.org/abs/2604.08075)：在可靠性约束下进行 Token 预算感知的在线路由。